# In class activity (November 5)
### Student(s): Caden Mendall

# Part 1: Load and Explore

In [2]:
import warnings
warnings.filterwarnings("ignore", category=Warning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score
)
wineData = load_wine()
X = pd.DataFrame(
    wineData.data,
    columns=wineData.feature_names
)
y = pd.Series(wineData.target)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print(X.head())
print(y.value_counts().sort_index())

Shape of X: (178, 13)
Shape of y: (178,)
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_dil

## Part 1 Questions:

Q1: How many observations are there in the dataset?  
There are 178 total samples / observations in this dataset


Q2: How many input features does each observation have?  
there are 13 input features for each observation, plus 1 output class


Q3: How many classes are there?  
There are 3 output categories / classes


Q4: Is this a regression or classification problem?  
Since the output is in discrete classes, this is a classification problem

# Part 2: Prepare the Data

In [11]:
# First: separate training data from the remaining data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Second: split the remaining data into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

# Third: scale the data so all classes are weighted equally

scaler = StandardScaler()

# Fit only on training data
X_train = scaler.fit_transform(X_train)

# Transform validation and test data using the same scaler
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert the data into PyTorch tensors:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_val = torch.tensor(y_val.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

Training: (106, 13) (106,)
Validation: (36, 13) (36,)
Testing: (36, 13) (36,)


## Part 2 Questions:

Q1: Why do we standardize features?  
Since some features have larger values than others, that can cause them to have more influence than smaller values. Standardizing them makes sure that all the features have the same relevance


Q2: Why do we fit the scaler using only training data  
If you fit it on all the data, then the scaler will contain information on the training. This means that some information on the test and validation sets could bleed into the model through the scaler


Q3: What is the purpose of the validation dataset?  
the validation dataset is meant to be used to tune the model


Q4: Why should we keep the test data seperate?  
the test data is meant to be the final evaluation, and the best safeguard against overfitting, which means it must be kept separate. If you use it to tune the data like a validation set, then you would often overfit

# Part 3: Design a Flexible ANN

In [14]:
class ClassificationNN(nn.Module):
    def __init__(self, hidden_sizes):
        super().__init__()

        layers = []
        input_size = X.shape[1]
        output_size = len(y.unique())  # one logit per class

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.Tanh())
            input_size = hidden_size

        # Raw logits: CrossEntropyLoss applies softmax internally
        layers.append(nn.Linear(input_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

## Part 3 Questions:

Q1: Why do we need an activation function?  
An activation function is what determines the value each neuron evaluates to. Tuning and improving these activation functions is the core functionaligy behind neural networks


Q2: How to calculate the number of input neurons in the first layer?  
The input neurons should be equivalent the number of input features (one neuron per feature)

# Part 4: Training the Model

In [ ]:
# Training function from model_eva.ipynb
def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=300,
    learning_rate=0.01,
    weight_decay=0.0
):
    loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()

        predictions = model(X_train)
        train_loss = loss_fn(predictions, y_train)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        # Evaluate validation loss without updating weights
        model.eval()
        with torch.no_grad():
            val_predictions = model(X_val)
            val_loss = loss_fn(val_predictions, y_val)

        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())

    return train_losses, val_losses

# train an example mode
torch.manual_seed(42)

example_model = ClassificationNN(hidden_sizes=[64, 32])

train_losses, val_losses = train_model(
    example_model,
    X_train,
    y_train,
    X_val,
    y_val,
)

# Part 5: Hyperparameter Tuning using Grid Search

## Part 5 Questions:

Q1: Which configuration has the lowest validation loss?  



Q2: Does the model with the lowest training loss also have the lowest validation loss?  



Q3: How does changing the number of hidden neurons affect model performance?  



Q4: What is the role of weight decay?  



Q5: Which configuration would you select, and why?  


# Part 6: Evaluate the Best Model